In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocessing

In [2]:
from datasets import load_from_disk

dataset = load_from_disk(
    PROJECT_ROOT/"datasets/landmark_recognition"
)

train_loader, val_loader, category_to_label, label_to_category = preprocessing(
    dataset=dataset,
    target_images=500,
    batch_size=32
)

Original training samples: 36463
Training samples: 32816
Validation samples: 3647

Balanced training samples: 25500
Number of classes: 51

Class distribution:
%C3%87ufut_Qale: 500
Akbar's_Tomb: 500
Akkerman_fortress: 500
Catedral_San_Sebasti%C3%A1n,_Cochabamba: 500
Corktown,_Toronto: 500
Douthat_State_Park: 500
Edinburgh_Castle: 500
Eiffel_Tower: 500
Episcopal_Diocese_of_Southwest_Florida: 500
Faisal_Mosque: 500
Feroz_Shah_Kotla: 500
Genoese_fortress_(Sudak): 500
Golden_Gate_Bridge: 500
Grand_Canyon: 500
Grote_of_Onze-Lieve-Vrouwekerk_(Breda): 500
Haghpat: 500
Haleakal%C4%81_National_Park: 500
Harichavank: 500
Hayravank_monastery: 500
Isa_Khan_Niyazi's_tomb: 500
Itmad-Ud-Daulah's_Tomb: 500
Jurassic_Coast: 500
Kecharis: 500
Khotyn_Fortress: 500
Lake_Como: 500
Ljubljana_Castle: 500
Lviv: 500
Madrid_R%C3%ADo: 500
Masada: 500
Mathura_Museum: 500
Matka_Canyon: 500
Media_contributed_by_the_ETH-Bibliothek: 500
Museum_of_Folk_Architecture_and_Ethnography_in_Pyrohiv: 500
Niagara_Falls: 500
Nieu

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from torchvision.models import ResNet50_Weights
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from datasets import load_from_disk
from src.preprocessing import preprocessing

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [5]:
weights = ResNet50_Weights.DEFAULT

model = models.resnet50(
    weights=weights
)

In [6]:
model.fc

Linear(in_features=2048, out_features=1000, bias=True)

```
   The original ResNet trained for classifiction of 1000 images 
   ours has only 51 so we have to change it 

In [7]:
NUM_CLASSES = 51

model.fc = nn.Linear(
    in_features=2048,
    out_features=NUM_CLASSES
)

In [8]:
# freezing previous layers and do allow training the classification layer

In [9]:
for param in model.parameters():
    param.requires_grad = False


for param in model.fc.parameters():
    param.requires_grad = True

In [10]:
model = model.to(device)

In [11]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

Total parameters: 23612531
Trainable parameters: 104499
Trainable percentage: 0.4426%


In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.fc.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=2
)

In [13]:

def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.eval()
    model.fc.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(train_loader, desc="Training")

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update trainable parameters
        optimizer.step()

        # Calculate statistics
        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "accuracy": f"{100 * correct_predictions / total_samples:.2f}%"
        })

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy


In [14]:

def validate_one_epoch(model, val_loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():

        progress_bar = tqdm(val_loader, desc="Validation")

        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            # Forward propagation
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            # Calculate statistics
            running_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            correct_predictions += (predictions == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "accuracy": f"{100 * correct_predictions / total_samples:.2f}%"
            })

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy


In [15]:
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = CHECKPOINT_DIR / "feature_extraction_latest.pth"

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_feature_extraction.pth"

NUM_EPOCHS = 20

In [16]:
def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_val_accuracy,
    history,
    checkpoint_path
):

    checkpoint = {
        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "best_val_accuracy": best_val_accuracy,

        "history": history,

        "category_to_label": category_to_label,

        "label_to_category": label_to_category
    }

    # Save to a temporary file first
    temp_path = checkpoint_path.with_suffix(".tmp")

    torch.save(checkpoint, temp_path)

    # Replace the previous checkpoint after saving succeeds
    temp_path.replace(checkpoint_path)

    print(f"Checkpoint saved after epoch {epoch}!")

In [17]:
def load_checkpoint(
    model,
    optimizer,
    scheduler,
    checkpoint_path,
    device
):

    if checkpoint_path.exists():

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        best_val_accuracy = checkpoint["best_val_accuracy"]

        history = checkpoint["history"]

        print("Previous checkpoint found!")

        print("Resuming from epoch:", start_epoch + 1)

        print("Best validation accuracy:", best_val_accuracy)

    else:

        start_epoch = 0

        best_val_accuracy = 0.0

        history = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }

        print("No checkpoint found. Starting fresh training.")

    return start_epoch, best_val_accuracy, history

In [18]:
start_epoch, best_val_accuracy, history = load_checkpoint(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    checkpoint_path=CHECKPOINT_PATH,
    device=device
)

Previous checkpoint found!
Resuming from epoch: 20
Best validation accuracy: 0.8851110501782287


In [19]:
for epoch in range(start_epoch, NUM_EPOCHS):

    print(f"\n{'='*60}")
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
    print(f"{'='*60}")

    # 1. Training
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # 2. Validation
    val_loss, val_accuracy = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device
    )

    # 3. Learning rate scheduling
    scheduler.step(val_loss)

    # 4. Store epoch metrics
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    # 5. Print epoch results
    print("\nEpoch Results")
    print("-" * 40)

    print(f"Train Loss       : {train_loss:.4f}")
    print(f"Train Accuracy   : {train_accuracy * 100:.2f}%")

    print(f"Validation Loss  : {val_loss:.4f}")
    print(f"Validation Acc.  : {val_accuracy * 100:.2f}%")

    print(f"Learning Rate    : {optimizer.param_groups[0]['lr']:.6f}")

    # 6. Save best model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "best_val_accuracy": best_val_accuracy,
                "category_to_label": category_to_label,
                "label_to_category": label_to_category
            },
            BEST_MODEL_PATH
        )

        print("\nBest model updated!")

    # 7. Save latest checkpoint after every epoch
    save_checkpoint(
        epoch=epoch + 1,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        best_val_accuracy=best_val_accuracy,
        history=history,
        checkpoint_path=CHECKPOINT_PATH
    )

print("\nTraining completed!")
print(f"Best Validation Accuracy: {best_val_accuracy * 100:.2f}%")


Epoch [20/20]


Training:   0%|          | 0/797 [00:00<?, ?it/s]

Validation:   0%|          | 0/114 [00:00<?, ?it/s]


Epoch Results
----------------------------------------
Train Loss       : 0.1398
Train Accuracy   : 96.62%
Validation Loss  : 0.3881
Validation Acc.  : 88.37%
Learning Rate    : 0.000010
Checkpoint saved after epoch 20!

Training completed!
Best Validation Accuracy: 88.51%
